###  **Environment Setup & Dependencies**



In [ ]:
# ==============================================================================
# 1. INITIAL SETUP & DEPENDENCIES
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Install Linux system dependencies (CDO for climate data processing)
!apt-get install -y cdo libnetcdf-dev

# Install Python packages for handling climate and weather data formats
# cdsapi: To download data from the Copernicus Climate Data Store
# xarray & cfgrib: To read, analyze, and manipulate GRIB/NetCDF files
!pip install cdsapi xarray[complete] cfgrib netCDF4 dask numpy

Mounted at /content/drive
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libnetcdf-dev is already the newest version (1:4.8.1-1).
libnetcdf-dev set to manually installed.
The following additional packages will be installed:
  at-spi2-core binfmt-support blt fonts-dejavu-core fonts-dejavu-extra
  fonts-droid-fallback fonts-font-awesome fonts-lato fonts-lyx fonts-noto-mono
  fonts-urw-base35 ghostscript gsettings-desktop-schemas javascript-common
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libcdi0
  libclang-cpp11 libdouble-conversion3 libdxflib3 libeccodes-data libeccodes0
  libeckit0d libevdev2 libfftw3-double3 libgs9 libgs9-common libgtk-3-0
  libgtk-3-bin libgtk-3-common libgudev-1.0-0 libidn12 libijs-0.35
  libimagequant0 libinput-bin libinput10 libjbig2dec0 libjs-sphinxdoc
  libjs-underscore liblbfgsb0 libllvm11 liblzf1 libmagics++-data libmagplus3v5
  libmd4c0 libmtdev1 libodc-0d libpfm4 libproj22 libqt5core5a lib

# **CERRA FILES**

### 2. **Directory Configuration**

In [ ]:
# ==============================================================================
# 2. DIRECTORY CONFIGURATION
# ==============================================================================
# Define the root directory path in Google Drive for the project
import os
BASE_PATH = "/content/drive/MyDrive/CERRA_processing"
os.makedirs(BASE_PATH, exist_ok=True)
os.chdir(BASE_PATH)

print(f"Environment is ready. Working directory set to: {BASE_PATH}")

Environment is ready. Working directory set to: /content/drive/MyDrive/CERRA_processing


### 3. **Target Grid Definition (Cylindrical Projection Remapping)**

**Scientific Context:**
CERRA natively adopts a **conical projection** optimized for high latitudes. Conversely, ERA5 uses a **cylindrical projection**. To conduct a direct, pixel-by-pixel comparative analysis or downscaling over lower-latitude regions (such as the Mediterranean/Greece), a coordinate transformation is required.

We define a standardized **Regular Cylindrical Lat/Lon grid** (`gridtype = lonlat`) at a high resolution of $0.05^\circ \times 0.05^\circ$ (~5km) to serve as the common destination grid for the remapping process.

In [ ]:
import os

# ==============================================================================
# 3. GRID DEFINITION (Cylindrical Transformation for Comparative Analysis)
# ==============================================================================
# Define parameters for a uniform Regular Lat/Lon grid over the study area.
# This serves as the target grid to resolve projection mismatches between ERA5 & CERRA.

grid_content = """gridtype = lonlat
xsize    = 256
ysize    = 256
xfirst   = 17.0
xinc     = 0.05
yfirst   = 33.0
yinc     = 0.05
"""

config_dir = "/content/drive/MyDrive/CERRA_processing/config"
os.makedirs(config_dir, exist_ok=True)

with open(f"{config_dir}/cyl_greece.txt", "w") as f:
    f.write(grid_content)

print(f"The file was successfully created at the path: {config_dir}/cyl_greece.txt")

The file was successfully created at the path: /content/drive/MyDrive/CERRA_processing/config/cyl_greece.txt


### 4.**Downloading Invariant Surface Constants (Orography & Land-Sea Mask)**

Before handling time-varying physical parameters, we download the invariant surface constants for our target period (2019). These files contain static topographical data (`orography`) and geographical boundaries (`land_sea_mask`) native to the CERRA Lambert Conformal Conic grid, which are vital for masking, verifying elevation adjustments, and establishing a baseline during the remapping process.*κείμενο σε πλάγια γραφή*

In [ ]:
import os
import cdsapi
import shutil

# ==============================================================================
# 4. DOWNLOAD STATIC LAND-SURFACE CONSTANTS
# ==============================================================================
import os
import cdsapi
import shutil

# API Key Configuration (Writes the credentials file to the home directory)
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write(f"url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: 85c978bc-363f-4b06-b7cd-8c9eb9898a21\n")

# Path configuration for the final output directory in Google Drive
drive_dir = "/content/drive/MyDrive/CERRA_processing/lambert_proj/single_levels/"
os.makedirs(drive_dir, exist_ok=True)

# List of the invariant spatial constants required for masking and remapping
static_vars = ["orography", "land_sea_mask"]

client = cdsapi.Client()

# Loop through each static variable using your exact request and download logic
for variable in static_vars:
    print(f"\n🚀 Starting process for static variable: {variable}")

    local_file = f"/content/{variable}.grib"
    final_drive_path = os.path.join(drive_dir, f"{variable}.grib")

    # CDS API Request Formulation
    dataset = "reanalysis-cerra-single-levels"
    request = {
        "variable": [variable],
        "level_type": "surface_or_atmosphere",
        "data_type": ["reanalysis"],
        "product_type": "analysis",
        "year": ["2019"],
        "month": ["01"],
        "day": ["01"],
        "time": ["00:00"],
        "data_format": "grib"
    }

    print(f"📥 Starting local download to Colab for {variable}...")
    client.retrieve(dataset, request).download(local_file)

    print(f"📂 Moving {variable} file to Google Drive...")
    shutil.move(local_file, final_drive_path)

    print(f"✅ DONE! File saved to: {final_drive_path}")

print("\n🏁 All static constants downloaded successfully.")


--- Starting process for year: 2014 ---
❌ ERROR for 2014: HTTPSConnectionPool(host='cds-beta.climate.copernicus.eu', port=443): Max retries exceeded with url: /api/retrieve/v1/processes/reanalysis-cerra-single-levels (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate (_ssl.c:1010)')))

--- Starting process for year: 2015 ---
❌ ERROR for 2015: HTTPSConnectionPool(host='cds-beta.climate.copernicus.eu', port=443): Max retries exceeded with url: /api/retrieve/v1/processes/reanalysis-cerra-single-levels (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate (_ssl.c:1010)')))

--- Starting process for year: 2016 ---
❌ ERROR for 2016: HTTPSConnectionPool(host='cds-beta.climate.copernicus.eu', port=443): Max retries exceeded with url: /api/retrieve/v1/processes/reanalysis-cerra-single-levels (Caused by SSLError(SSLCertVerificationError

###  **Multi-Year Time-Series Extraction (2010 - 2021)**

This unified block automates the download of the 3-hourly $2\text{m}$ temperature dataset (`2m_temperature`) for the entire historical window from 2010 to 2021. To optimize execution and prevent Google Colab from running out of disk space, the data is processed in chronological annual batches. Each file is staged locally in the environment before being safely migrated to Google Drive.

In [ ]:
import os
import cdsapi
import shutil

# 1. NEW API Configuration for CDS-Beta
# Note: The URL must be the beta one for current CERRA requests
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write("key: 85c978bc-363f-4b06-b7cd-8c9eb9898a21\n") # Using your key

client = cdsapi.Client()

# 2. Path Configuration
DRIVE_BASE_PATH = "/content/drive/MyDrive/CERRA_Data"
os.makedirs(DRIVE_BASE_PATH, exist_ok=True)

# 3. Years List
years = ["2010","2011","2012","2013","2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021"]
dataset = "reanalysis-cerra-single-levels"

# 4. Download Loop
for year in years:
    print(f"\n--- Starting process for year: {year} ---")

    local_file = f"temp_{year}.grib"
    final_drive_path = os.path.join(DRIVE_BASE_PATH, f"cerra_temp_{year}.grib")

    request = {
        "variable": ["2m_temperature"],
        "level_type": ["surface_or_atmosphere"],
        "data_type": ["reanalysis"],
        "product_type": ["analysis"], # Wrapped in list for the new API
        "year": [year],
        "month": ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"],
        "day": [
            "01", "02", "03", "04", "05", "06", "07", "08", "09", "10",
            "11", "12", "13", "14", "15", "16", "17", "18", "19", "20",
            "21", "22", "23", "24", "25", "26", "27", "28", "29", "30", "31"
        ],
        "time": ["00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00"],
        "data_format": "grib"
    }

    try:
        print(f"Downloading {year} from CDS-Beta...")
        client.retrieve(dataset, request).download(local_file)

        print(f"Moving {year} to Google Drive...")
        shutil.move(local_file, final_drive_path)
        print(f"✅ SUCCESS: {final_drive_path}")

    except Exception as e:
        print(f"❌ ERROR for {year}: {e}")

print("\n--- ALL YEARS COMPLETED ---")



--- Starting process for year: 2014 ---


2026-05-02 08:22:32,450 INFO Request ID is 230ed6f6-bdfd-4683-a0e3-f715f28355e8
INFO:ecmwf.datastores.legacy_client:Request ID is 230ed6f6-bdfd-4683-a0e3-f715f28355e8
2026-05-02 08:22:32,603 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted


KeyboardInterrupt: 

###  **CDO System Initialization & Remapping**

In [ ]:

# ==============================================================================
# Purge mismatched repositories and install clean CDO binaries
!sudo add-apt-repository --remove ppa:ubuntugis/ppa -y
!apt-get update -qq
!apt-get install -y --fix-missing cdo
!ulimit -s unlimited

PPA publishes dbgsym, you may need to include 'main/debug' component
Repository: 'deb https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/ jammy main'
Description:
Official stable UbuntuGIS packages.


More info: https://launchpad.net/~ubuntugis/+archive/ubuntu/ppa
Removing repository.
Disabling deb entry in /etc/apt/sources.list.d/ubuntugis-ubuntu-ppa-jammy.list
Removing disabled deb entry from /etc/apt/sources.list.d/ubuntugis-ubuntu-ppa-jammy.list
Removing disabled deb-src entry from /etc/apt/sources.list.d/ubuntugis-ubuntu-ppa-jammy.list
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinoi

In [ ]:
import os
import subprocess

# ==============================================================================
#  CDO TEMPERATURE TIME-SERIES REMAPPING (GRIB TO NETCDF4)
# ==============================================================================
GRID_FILE = "/content/drive/MyDrive/CERRA_processing/config/cyl_greece.txt"
INPUT_DIR = "/content/drive/MyDrive/CERRA_Data"
OUTPUT_DIR = "/content/drive/MyDrive/CERRA_processing/latlon_proj_Greece/remapped"

if not os.path.exists(GRID_FILE):
    os.system(f"ls {os.path.dirname(GRID_FILE)}")

if os.path.exists(GRID_FILE):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    all_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".grib")]
    print(f" There are {len(all_files)} files....")

    for f in sorted(all_files):
        in_p = os.path.join(INPUT_DIR, f)
        clean_name = f.replace("cerra_", "").replace(".grib", ".nc")
        out_p = os.path.join(OUTPUT_DIR, clean_name)

        if not os.path.exists(out_p):
            print(f"\n>>> process: {f} -> {clean_name}")
            !cdo -f nc4 -z zip_4 remapbil,{GRID_FILE} {in_p} {out_p}
        else:
            print(f"Skip: {clean_name}")
else:
    print("Refresh ")

print("\n--- DONE ---")

 There are 8 files....

>>> process: cerra_temp_2014.grib -> temp_2014.nc
cdo    remapbil: Bilinear weights from curvilinear (1069x1069) to lonlat (256x256) grid
cdo    remapbil:   0%  0%  1%  2%  3%  4%  5%  6%  7%  8%  9% 10% 11% 12% 13% 14% 15% 16% 17% 18% 19% 20% 21% 22% 23% 24% 25% 26% 27% 28% 29% 30% 31% 32% 33% 34% 35% 36% 37% 38% 39% 40% 41% 42% 43% 44% 45% 46% 47% 48% 49% 50% 51% 52% 53% 54% 55% 56% 57% 58% 59% 60% 61% 62% 63% 64% 65% 66% 67% 68% 69% 70% 71% 72% 73% 74% 75% 76% 77% 78% 79% 80% 81% 82% 83% 84% 85% 86% 87% 88% 89% 90% 91% 92% 93% 94% 95% 96% 97% 98% 99%100%   

In [ ]:
# ==============================================================================
# APPROACH A: CLIPPING AFTER BILINEAR INTERPOLATION (RECOMMENDED)
# ==============================================================================
import os

GRID_FILE = "/content/drive/MyDrive/CERRA_processing/config/cyl_greece.txt"
INPUT_FILE = "/content/drive/MyDrive/CERRA_processing/lambert_proj/single_levels/orography.grib"
OUTPUT_DIR = "/content/drive/MyDrive/CERRA_processing/latlon_proj_Greece/remapped"
OUTPUT_A = os.path.join(OUTPUT_DIR, "orography_clipped_after.nc")

if os.path.exists(GRID_FILE) and os.path.exists(INPUT_FILE):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if not os.path.exists(OUTPUT_A):
        print(">>> Executing Approach A: Remap first, then clip negatives...")
        # Pipeline: Raw GRIB -> remapbil -> set negative range to missing -> convert missing to 0
        !cdo -f nc4 -z zip_4 -setmisstoc,0 -setrtomiss,-999,0 -remapbil,{GRID_FILE} {INPUT_FILE} {OUTPUT_A}
        print(f"✅ Approach A Complete: {OUTPUT_A}")
    else:
        print("Skip: Output for Approach A already exists.")

>>> Executing Approach A: Remap first, then clip negatives...
cdo(1) setrtomiss: Process started
cdo(2) remapbil: Process started
cdo(2) remapbil: Bilinear weights from curvilinear (1069x1069) to lonlat (256x256) grid
cdo(2) remapbil:   0%  0%  1%  2%  3%  4%  5%  6%  7%  8%  9% 10% 11% 12% 13% 14% 15% 16% 17% 18% 19% 20% 21% 22% 23% 24% 25% 26% 27% 28% 29% 30% 31% 32% 33% 34% 35% 36% 37% 38% 39% 40% 41% 42% 43% 44% 45% 46% 47% 48% 49% 50% 51% 52% 53% 54% 55% 56% 57% 58% 59% 60% 61% 62% 63% 64% 65% 66% 67% 68% 69% 70% 71% 72% 73% 74% 75% 76% 77% 78% 79% 80% 81% 82% 83% 84% 85% 86% 87% 88% 89% 90% 91% 92% 93% 94%

In [ ]:
import os
import subprocess

GRID_FILE = "/content/drive/MyDrive/CERRA_processing/config/cyl_greece.txt"
INPUT_DIR = "/content/drive/MyDrive/CERRA_processing/lambert_proj/single_levels/land_sea_mask.grib"
OUTPUT_DIR = "/content/drive/MyDrive/CERRA_processing/latlon_proj_Greece/remapped"

if not os.path.exists(GRID_FILE):
    os.system(f"ls {os.path.dirname(GRID_FILE)}")

if os.path.exists(GRID_FILE):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    all_files = [os.path.basename(INPUT_DIR)]
    print(f" There are {len(all_files)} files....")

    for f in sorted(all_files):
        in_p = INPUT_DIR
        clean_name = f.replace("cerra_", "").replace(".grib", ".nc")
        out_p = os.path.join(OUTPUT_DIR, clean_name)

        if not os.path.exists(out_p):
            print(f"\n>>> process: {f} -> {clean_name}")
            !cdo -f nc4 -z zip_4 remapnn,{GRID_FILE} {in_p} {out_p}
        else:
            print(f"Skip: {clean_name} (already exist)")

else:
    print("Refresh ")

print("\n--- DONE ---")

 There are 1 files....

>>> process: land_sea_mask.grib -> land_sea_mask.nc
cdo    remapnn: Nearest neighbor weights from curvilinear (1069x1069) to lonlat (256x256) grid
cdo    remapnn:   0%  0%  1%  2%  3%  4%  5%  6%  7%  8%  9% 10% 11% 12% 13% 14% 15% 16% 17% 18% 19% 20% 21% 22% 23% 24% 25% 26% 27% 28% 29% 30% 31% 32% 33% 34% 35% 36% 37% 38% 39% 40% 41% 42% 43% 44% 45% 46% 47% 48% 49% 50% 51% 52% 53% 54% 55% 56% 57% 58% 59% 60% 61% 62% 63% 64% 65% 66% 67% 68% 69% 70% 71% 72% 73% 74% 75% 76% 77% 78% 79% 80% 81% 82% 83% 84% 85% 86% 87% 88% 89% 90% 91% 92% 93% 94% 95% 96% 97% 98% 99%100%

### **Metadata & Quality Inspector**





In [ ]:
# Exploring files
!pip install --upgrade netCDF4 cftime xarray

import xarray as xr
import os
import glob
import numpy as np

# 1. Folder path holding remapped NetCDF files
input_folder = "/content/drive/MyDrive/CERRA_processing/latlon_proj_Greece/remapped"

# 2. Collect and sort all processed NetCDF outputs
nc_files = sorted(glob.glob(os.path.join(input_folder, "*.nc")))

print(f"🔎 Found {len(nc_files)} files to inspect.\n")

for file_path in nc_files:
    file_name = os.path.basename(file_path)
    print(f"{'='*75}")
    print(f"📄 INSPECTING METADATA: {file_name}")
    print(f"{'='*75}")

    try:
        # Open dataset without loading array into memory (lazy loading)
        ds = xr.open_dataset(file_path)

        # Dynamic variable identification using keywords
        var_key = None
        for var in ds.data_vars:
            # Check for temperature variations (2t, t2m, t)
            if any(k in var.lower() for k in ['2t', 't2m', 'temp']) or var.lower() == 't':
                var_key = var
                break
            # Check for orography/elevation variations (orog, z, elev)
            elif any(k in var.lower() for k in ['orog', 'elev', 'orography']) or var.lower() == 'z':
                var_key = var
                break
            # Check for land-sea mask
            elif 'mask' in var.lower() or 'lsm' in var.lower():
                var_key = var
                break

        # Fallback if no specific match, pick the first available data variable
        if not var_key and list(ds.data_vars):
            var_key = list(ds.data_vars)[0]

        # Metadata Conventions Check
        print(f"✅ CF Conventions: {ds.attrs.get('Conventions', 'N/A')}")

        # --- UNITS & VARIABLE INFO ---
        print("\n📊 VARIABLE DATA & ATTRIBUTES:")
        for var in ds.data_vars:
            unit = ds[var].attrs.get('units', 'No units defined')
            long_name = ds[var].attrs.get('long_name', 'No description')
            print(f"   -> Variable: {var:10} | Units: {unit:12} | Info: {long_name}")
        print("-" * 40)

        # --- DIMENSIONS & SHAPE ---
        if var_key:
            print(f"📐 TARGET FIELD:          {var_key}")
            print(f"📐 SHAPE (Dimensions):    {ds[var_key].shape} -> {list(ds[var_key].dims)}")
            print(f"🔢 TOTAL GRID POINTS:     {ds[var_key].size:,}")

        # --- TIME BOUNDS ---
        if 'time' in ds.coords:
            ds = ds.sortby('time')
            start_t = ds.time.values[0]
            end_t = ds.time.values[-1]
            print(f"📅 TIME HORIZON:          {np.datetime_as_string(start_t, unit='h')} to {np.datetime_as_string(end_t, unit='h')} ({len(ds.time)} steps)")
        else:
            print("📅 TIME HORIZON:          Static Spatial Invariant (No time axis)")

        # --- GEOGRAPHIC EXTENT ---
        lat_name = 'lat' if 'lat' in ds.coords else 'latitude'
        lon_name = 'lon' if 'lon' in ds.coords else 'longitude'

        if lat_name in ds.coords and lon_name in ds.coords:
            print(f"📍 LATITUDE RANGE:       {float(ds[lat_name].min()):.3f}°N to {float(ds[lat_name].max()):.3f}°N")
            print(f"📍 LONGITUDE RANGE:      {float(ds[lon_name].min()):.3f}°E to {float(ds[lon_name].max()):.3f}°E")

        # --- PHYSICAL RANGE INTEGRITY CHECK ---
        if var_key:
            v_min = float(ds[var_key].min())
            v_max = float(ds[var_key].max())
            v_mean = float(ds[var_key].mean())
            print(f"🌡️ DATA VALUE METRICS:   Min: {v_min:.2f} | Max: {v_max:.2f} | Mean: {v_mean:.2f}")

        ds.close()

    except Exception as e:
        print(f"❌ ERROR processing file {file_name}: {e}")

    print("\n")

print("--- ALL NC FILES SUCCESSFULLY VERIFIED ---")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.0 MB/s eta 0:00:00
  Attempting uninstall: cftime
    Found existing installation: cftime 1.5.2
    Uninstalling cftime-1.5.2:
      Successfully uninstalled cftime-1.5.2
  Attempting uninstall: xarray
    Found existing installation: xarray 2025.12.0
    Uninstalling xarray-2025.12.0:
      Successfully uninstalled xarray-2025.12.0
🔎 Found 14 files to inspect.

📄 INSPECTING METADATA: land_sea_mask.nc
✅ CF Conventions: CF-1.6

📊 VARIABLE DATA & ATTRIBUTES:
   -> Variable: lsm        | Units: (0 - 1)      | Info: Land-sea mask
----------------------------------------
📐 TARGET FIELD:          lsm
📐 SHAPE (Dimensions):    (1, 256, 256) -> ['time', 'lat', 'lon']
🔢 TOTAL GRID POINTS:     65,536
📅 TIME HORIZON:          2019-01-01T00 to 2019-01-01T00 (1 steps)
📍 LATITUDE RANGE:       33.000°N to 45.750°N
📍 LONGITUDE RANGE:      17.000°E to 29.750°E
🌡️ DATA VA

### **Preprocess Cerra**

In [ ]:
!pip install cftime==1.6.4 netCDF4 xarray

In [ ]:
# ==============================================================================
#  FINAL DATA STANDARDIZATION & CF-COMPLIANCE PIPELINE
# ==============================================================================
import os
import glob
import xarray as xr
import pandas as pd
import numpy as np

# Config Directories
input_dir = "/content/drive/MyDrive/CERRA_processing/latlon_proj_Greece/remapped"
output_dir = "/content/drive/MyDrive/CERRA_processing/final_standardized"
os.makedirs(output_dir, exist_ok=True)

years = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]

# ------------------------------------------------------------------------------
# STEP 1: PROCESS METEOROLOGICAL VARIABLES (TEMPERATURE)
# ------------------------------------------------------------------------------
t2m_encoding = {
    't2m': {
        'zlib': True,
        'complevel': 5,
        'chunksizes': (1, 256, 256)  # Optimized for time-series slicing
    },
    'time': {
        'units': 'hours since 1900-01-01 00:00:00',
        'calendar': 'proleptic_gregorian'
    }
}

for year in years:
    file_path = os.path.join(input_dir, f"temp_{year}.nc")
    if not os.path.exists(file_path):
        print(f"⚠️ Warning: temp_{year}.nc not found. Skipping...")
        continue

    print(f"🔄 Standardizing Temperature for year: {year}...")

    # Load and force time chronological order
    ds = xr.open_dataset(file_path).sortby('time')
    ds = ds.squeeze(drop=True)

    # Standardize names safely
    rename_dict = {'2t': 't2m', 'lat': 'latitude', 'lon': 'longitude'}
    ds = ds.rename({k: v for k, v in rename_dict.items() if k in ds.variables or k in ds.coords})

    # Strict Leap-Year Quality Control check
    is_leap = (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)
    samples = 2928 if is_leap else 2920

    if len(ds.time) != samples:
        raise ValueError(f"🚨 Data Integrity Error: Year {year} has {len(ds.time)} steps instead of {samples}!")

    save_path = os.path.join(output_dir, f"CERRA_Greece_{year}.nc")
    ds.to_netcdf(save_path, encoding=t2m_encoding)
    print(f"✅ Saved Standardized Year {year}! Shape: {ds.t2m.shape}")

print("-" * 50)

# ------------------------------------------------------------------------------
# STEP 2: PROCESS STATIC INVARIANT 1 (OROGRAPHY) - USING CLIPPED FILE
# ------------------------------------------------------------------------------
# FIX: Points to your curated 'orography_clipped_after.nc' to avoid ocean artifacts
static_path = os.path.join(input_dir, "orography_clipped_after.nc")

if os.path.exists(static_path):
    print("🔄 Standardizing Static Orography map...")
    ds_s = xr.open_dataset(static_path).squeeze(drop=True)

    # Dynamic detection of raw orography variable name
    raw_orog_var = [v for v in ds_s.data_vars if 'orog' in v or 'z' in v or 'elev' in v][0]

    # Rename variables and dimensions to standard template
    s_rename = {raw_orog_var: 'orography', 'lat': 'latitude', 'lon': 'longitude'}
    ds_s = ds_s.rename({k: v for k, v in s_rename.items() if k in ds_s.variables or k in ds_s.coords})

    if 'time' in ds_s.coords:
        ds_s = ds_s.drop_vars('time')

    # CF Metadata injection
    ds_s.attrs['Conventions'] = 'CF-1.8'
    ds_s.attrs['title'] = 'Standardized Orography for Downscaling'
    ds_s['orography'].attrs = {
        'standard_name': 'surface_altitude',
        'long_name': 'Surface Orography Elevation',
        'units': 'm',
    }

    static_enc = {
        'orography': {
            'zlib': True,
            'complevel': 5,
            'chunksizes': (256, 256) # Matched grid layout
        }
    }
    ds_s.to_netcdf(os.path.join(output_dir, "CERRA_orography_static.nc"), encoding=static_enc)
    print("🚀 Static Orography cleaned and saved successfully!")
else:
    print(f"❌ Error: Clipped orography file not found at {static_path}!")

print("-" * 50)

# ------------------------------------------------------------------------------
# STEP 3: PROCESS STATIC INVARIANT 2 (LAND-SEA MASK)
# ------------------------------------------------------------------------------
lsm_path = os.path.join(input_dir, "land_sea_mask.nc")

if os.path.exists(lsm_path):
    print("🔄 Standardizing Land-Sea Mask fraction...")
    ds_lsm = xr.open_dataset(lsm_path).squeeze(drop=True)

    if 'lsm' in ds_lsm.data_vars:
        ds_lsm['lsm'] = ds_lsm['lsm'].astype(np.float32)

    lsm_rename = {'lsm': 'land_mask', 'lat': 'latitude', 'lon': 'longitude'}
    ds_lsm = ds_lsm.rename({k: v for k, v in lsm_rename.items() if k in ds_lsm.variables or k in ds_lsm.coords})

    if 'time' in ds_lsm.coords:
        ds_lsm = ds_lsm.drop_vars('time')

    ds_lsm.attrs['Conventions'] = 'CF-1.8'
    ds_lsm.attrs['title'] = 'Standardized Land-Sea Mask for Downscaling'
    ds_lsm['land_mask'].attrs = {
        'standard_name': 'land_area_fraction',
        'long_name': 'Land-sea area fraction',
        'units': '1',
        'comment': 'Derived from CERRA land-sea mask. Values represent the fraction of land in each cell (0=Ocean, 1=Land).'
    }

    lsm_encoding = {
        'land_mask': {
            'zlib': True,
            'complevel': 5,
            'dtype': 'float32',
            'chunksizes': (256, 256) # Matched grid layout
        }
    }

    ds_lsm.to_netcdf(os.path.join(output_dir, "CERRA_land_sea_mask.nc"), encoding=lsm_encoding)
    print(f"🚀 Static Land-Sea Mask saved! Shape: {ds_lsm['land_mask'].shape}")
else:
    print(f"❌ Error: Land-Sea Mask file not found at {lsm_path}!")

print("\n==================================================================")
# Print check to verify all standardized outputs are safely in place
print("🎉 ALL FILES STANDARDIZED AND SAVED IN 'final_standardized' FOLDER!")
print(os.listdir(output_dir))
print("==================================================================")

🔄 Standardizing Temperature for year: 2010...
✅ Saved Standardized Year 2010! Shape: (2920, 256, 256)
🔄 Standardizing Temperature for year: 2011...
✅ Saved Standardized Year 2011! Shape: (2920, 256, 256)
🔄 Standardizing Temperature for year: 2012...
✅ Saved Standardized Year 2012! Shape: (2928, 256, 256)
🔄 Standardizing Temperature for year: 2013...
✅ Saved Standardized Year 2013! Shape: (2920, 256, 256)
🔄 Standardizing Temperature for year: 2014...
✅ Saved Standardized Year 2014! Shape: (2920, 256, 256)
🔄 Standardizing Temperature for year: 2015...
✅ Saved Standardized Year 2015! Shape: (2920, 256, 256)
🔄 Standardizing Temperature for year: 2016...
✅ Saved Standardized Year 2016! Shape: (2928, 256, 256)
🔄 Standardizing Temperature for year: 2017...
✅ Saved Standardized Year 2017! Shape: (2920, 256, 256)
🔄 Standardizing Temperature for year: 2018...
✅ Saved Standardized Year 2018! Shape: (2920, 256, 256)
🔄 Standardizing Temperature for year: 2019...
✅ Saved Standardized Year 2019! Shap

/tmp/ipykernel_6248/1638901032.py:97: UserWarning: Unlimited dimension(s) {'time'} declared in 'dataset.encoding', but not part of current dataset dimensions. Consider removing {'time'} from 'dataset.encoding'.
  ds_s.to_netcdf(os.path.join(output_dir, "CERRA_orography_static.nc"), encoding=static_enc)


🚀 Static Orography cleaned and saved successfully!
--------------------------------------------------
🔄 Standardizing Land-Sea Mask fraction...
🚀 Static Land-Sea Mask saved! Shape: (256, 256)

🎉 ALL FILES STANDARDIZED AND SAVED IN 'final_standardized' FOLDER!
['.ipynb_checkpoints', 'CERRA_Greece_2010.nc', 'CERRA_Greece_2011.nc', 'CERRA_Greece_2012.nc', 'CERRA_Greece_2013.nc', 'CERRA_Greece_2014.nc', 'CERRA_Greece_2015.nc', 'CERRA_Greece_2016.nc', 'CERRA_Greece_2017.nc', 'CERRA_Greece_2018.nc', 'CERRA_Greece_2019.nc', 'CERRA_Greece_2020.nc', 'CERRA_Greece_2021.nc', 'CERRA_orography_static.nc', 'CERRA_land_sea_mask.nc']


/tmp/ipykernel_6248/1638901032.py:140: UserWarning: Unlimited dimension(s) {'time'} declared in 'dataset.encoding', but not part of current dataset dimensions. Consider removing {'time'} from 'dataset.encoding'.
  ds_lsm.to_netcdf(os.path.join(output_dir, "CERRA_land_sea_mask.nc"), encoding=lsm_encoding)


# **ERA5 FILES**





### **ERA5 Single Levels Data Downloader**

In [ ]:
# ==============================================================================
# ERA5 DATA RETRIEVAL: SURFACE & SINGLE LEVELS (3-HOURLY RESOLUTION)
# ==============================================================================
import os
import cdsapi

# 1. Credentials Configuration
# Writing the API URL and private key to the hidden config file required by cdsapi
with open('/root/.cdsapirc', 'w') as f:
    f.write('url: https://cds.climate.copernicus.eu/api\n')
    f.write('key: 85c978bc-363f-4b06-b7cd-8c9eb9898a21\n')

# Initialize the ECMWF Climate Data Store API Client
c = cdsapi.Client()

# 2. Storage Setup
output_dir = "/content/drive/MyDrive/ERA5_data"
os.makedirs(output_dir, exist_ok=True)

# Define the historical temporal range (Adjust list if expanding historical baseline)
years = ['2010', '2011', '2012', '2013','2014','2015','2016','2017','2018','2019','2020','2021']

# 3. Execution Loop per Year
for year in years:
    output_filename = os.path.join(output_dir, f"era5_raw_{year}.nc")

    print(f"\n--- Starting download workflow for Year: {year} ---")

    # Local Cache Check: Prevent re-downloading existing NetCDF files
    if os.path.exists(output_filename):
        print(f"File {output_filename} already exists locally. Skipping request...")
        continue

    try:
        # Submit asynchronous data request to the Copernicus Server
        c.retrieve(
            'reanalysis-era5-single-levels',
            {
                'product_type': 'reanalysis',
                'format': 'netcdf',               # Exporting directly to NetCDF4 format
                'variable': [
                    '10m_u_component_of_wind',    # Zonal wind component (East-West)
                    '10m_v_component_of_wind',    # Meridional wind component (North-South)
                    '2m_temperature',             # Air temperature at 2 meters altitude

                ],
                'year': year,
                'month': [
                    '01', '02', '03', '04', '05', '06',
                    '07', '08', '09', '10', '11', '12',
                ],
                'day': [
                    f"{d:02d}" for d in range(1, 32) # Standardizing day format to 2 digits (01-31)
                ],
                'time': [
                    '00:00', '03:00', '06:00', '09:00',
                    '12:00', '15:00', '18:00', '21:00', # 3-hourly sampling rate matching CERRA steps
                ],
                # Spatial Subsetting Bounding Box: [North, West, South, East]
                # Exactly matched to the Greek geographical domain defined in CERRA processing
                'area': [45.75, 17, 33, 29.75],
            },
            output_filename)
        print(f"✅ Download Successful: {output_filename}")

    except Exception as e:
        print(f"❌ API Error encountered for Year {year}: {e}")

print("\n--- ALL SURFACE LEVEL DOWNLOADS COMPLETED ---")


In [ ]:
# ==============================================================================
# ERA5 DATA RETRIEVAL: SURFACE & SINGLE LEVELS (3-HOURLY RESOLUTION)
# Dewpoint temperature at 2 meters altitude
# ==============================================================================
import os
import cdsapi

# 1. Credentials Configuration
# Writing the API URL and private key to the hidden config file required by cdsapi
with open('/root/.cdsapirc', 'w') as f:
    f.write('url: https://cds.climate.copernicus.eu/api\n')
    f.write('key: 85c978bc-363f-4b06-b7cd-8c9eb9898a21\n')

# Initialize the ECMWF Climate Data Store API Client
c = cdsapi.Client()

# 2. Storage Setup
output_dir = "/content/drive/MyDrive/ERA5_data"
os.makedirs(output_dir, exist_ok=True)

# Define the historical temporal range (Adjust list if expanding historical baseline)
years = ['2010', '2011', '2012', '2013','2014','2015','2016','2017','2018','2019','2020','2021']

# 3. Execution Loop per Year
for year in years:
    output_filename = os.path.join(output_dir, f"era5_raw_{year}.nc")

    print(f"\n--- Starting download workflow for Year: {year} ---")

    # Local Cache Check: Prevent re-downloading existing NetCDF files
    if os.path.exists(output_filename):
        print(f"File {output_filename} already exists locally. Skipping request...")
        continue

    try:
        # Submit asynchronous data request to the Copernicus Server
        c.retrieve(
            'reanalysis-era5-single-levels',
            {
                'product_type': 'reanalysis',
                'format': 'netcdf',               # Exporting directly to NetCDF4 format
                'variable': [
                    '2m_dewpoint_temperature',    # Dewpoint temperature at 2 meters altitude

                ],
                'year': year,
                'month': [
                    '01', '02', '03', '04', '05', '06',
                    '07', '08', '09', '10', '11', '12',
                ],
                'day': [
                    f"{d:02d}" for d in range(1, 32) # Standardizing day format to 2 digits (01-31)
                ],
                'time': [
                    '00:00', '03:00', '06:00', '09:00',
                    '12:00', '15:00', '18:00', '21:00', # 3-hourly sampling rate matching CERRA steps
                ],
                # Spatial Subsetting Bounding Box: [North, West, South, East]
                # Exactly matched to the Greek geographical domain defined in CERRA processing
                'area': [45.75, 17, 33, 29.75],
            },
            output_filename)
        print(f"✅ Download Successful: {output_filename}")

    except Exception as e:
        print(f"❌ API Error encountered for Year {year}: {e}")

print("\n--- ALL SURFACE LEVEL DOWNLOADS COMPLETED ---")




### **ERA5 Pressure Levels Data Downloader (850 hPa)**

In [ ]:
# ==============================================================================
# ERA5 DATA RETRIEVAL: UPPER-AIR PRESSURE LEVELS (T850 AT 3-HOURLY STEP)
# ==============================================================================
import os
import cdsapi

# 1. Credentials Configuration
with open('/root/.cdsapirc', 'w') as f:
    f.write('url: https://cds.climate.copernicus.eu/api\n')
    f.write('key: 85c978bc-363f-4b06-b7cd-8c9eb9898a21\n')

# Initialize the API Client
c = cdsapi.Client()

# 2. Storage Setup
output_dir = "/content/drive/MyDrive/ERA5_data"
os.makedirs(output_dir, exist_ok=True)

# Extended historical array covering full validation and training epochs
years = ['2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021']

# 3. Execution Loop per Year
for year in years:
    output_filename = os.path.join(output_dir, f"era5_temp850_{year}.nc")

    print(f"\n--- Starting upper-air download workflow for Year: {year} ---")

    # Local Cache Check
    if os.path.exists(output_filename):
        print(f"File {output_filename} already exists locally. Skipping request...")
        continue

    try:
        # Submit query to the upper-air pressure levels dataset
        c.retrieve(
            'reanalysis-era5-pressure-levels',
            {
                'product_type': 'reanalysis',
                'format': 'netcdf',
                'variable': [
                    'temperature'                 # Free-atmosphere air temperature
                ],
                'year': year,
                'month': [
                    '01', '02', '03', '04', '05', '06',
                    '07', '08', '09', '10', '11', '12',
                ],
                'day': [
                    f"{d:02d}" for d in range(1, 32)
                ],
                'time': [
                    '00:00', '03:00', '06:00', '09:00',
                    '12:00', '15:00', '18:00', '21:00', # Snychronized temporal grid
                ],
                'pressure_level': [
                    '850'                         # Isobaric level at ~1.5 km height (Boundary Layer top)
                ],
                # Identical domain subsetting for seamless spatial colocation
                'area': [45.75, 17, 33, 29.75],
            },
            output_filename)
        print(f"✅ Download Successful: {output_filename}")

    except Exception as e:
        print(f"❌ API Error encountered for Year {year}: {e}")

print("\n--- ALL UPPER-AIR DOWNLOADS COMPLETED ---")

In [ ]:
# ==============================================================================
# RAW MERGING OF ALL ERA5 FILES PER YEAR
# ==============================================================================
import os
import xarray as xr

# Directory where your files are located
era5_dir = "/content/drive/MyDrive/ERA5_data"

# Full list of years
years = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]

for year in years:
    print(f"🔄 Merging raw files for year: {year}...")

    # Define paths for the 3 files of the current year
    raw_path = os.path.join(era5_dir, f"era5_raw_{year}.nc")
    dew_path = os.path.join(era5_dir, f"era5_2dew_{year}.nc")
    t850_path = os.path.join(era5_dir, f"era5_temp850_{year}.nc")

    # Check if all 3 files exist before merging
    if os.path.exists(raw_path) and os.path.exists(dew_path) and os.path.exists(t850_path):

        # Open the 3 files exactly as they are
        ds_raw = xr.open_dataset(raw_path)
        ds_dew = xr.open_dataset(dew_path)
        ds_t850 = xr.open_dataset(t850_path)

        # Direct merge into one dataset
        ds_merged = xr.merge([ds_raw, ds_dew, ds_t850])

        # Save the combined raw file
        output_filename = os.path.join(era5_dir, f"era5_combined_raw_{year}.nc")
        ds_merged.to_netcdf(output_filename)

        print(f"✅ Year {year} completed! Saved as: {output_filename}")
    else:
        print(f"⚠️ Skipping {year} because one or more files were not found.")

print("\n🎉 ALL YEARS MERGED SUCCESSFULLY!")

🔄 Merging raw files for year: 2010...


/tmp/ipykernel_6248/2952143397.py:30: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.merge([ds_raw, ds_dew, ds_t850])
/tmp/ipykernel_6248/2952143397.py:30: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.merge([ds_raw, ds_dew, ds_t850])


✅ Year 2010 completed! Saved as: /content/drive/MyDrive/ERA5_data/era5_combined_raw_2010.nc
🔄 Merging raw files for year: 2011...


/tmp/ipykernel_6248/2952143397.py:30: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.merge([ds_raw, ds_dew, ds_t850])
/tmp/ipykernel_6248/2952143397.py:30: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.merge([ds_raw, ds_dew, ds_t850])


✅ Year 2011 completed! Saved as: /content/drive/MyDrive/ERA5_data/era5_combined_raw_2011.nc
🔄 Merging raw files for year: 2012...


...

✅ Year 2019 completed! Saved as: /content/drive/MyDrive/ERA5_data/era5_combined_raw_2019.nc
🔄 Merging raw files for year: 2020...


/tmp/ipykernel_6248/2952143397.py:30: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.merge([ds_raw, ds_dew, ds_t850])
/tmp/ipykernel_6248/2952143397.py:30: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.merge([ds_raw, ds_dew, ds_t850])


✅ Year 2020 completed! Saved as: /content/drive/MyDrive/ERA5_data/era5_combined_raw_2020.nc
🔄 Merging raw files for year: 2021...


/tmp/ipykernel_6248/2952143397.py:30: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.merge([ds_raw, ds_dew, ds_t850])
/tmp/ipykernel_6248/2952143397.py:30: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_merged = xr.merge([ds_raw, ds_dew, ds_t850])


✅ Year 2021 completed! Saved as: /content/drive/MyDrive/ERA5_data/era5_combined_raw_2021.nc

🎉 ALL YEARS MERGED SUCCESSFULLY!


### **Metadata & Quality Inspector**





In [ ]:
# ==============================================================================
# ERA5 MULTI-FILE METADATA INSPECTOR (PROPER TIME & VAR DETECTION)
# ==============================================================================
import xarray as xr
import os
import glob

input_folder = "/content/drive/MyDrive/ERA5_data"

# Scan folder for NetCDF files
nc_files = sorted(glob.glob(os.path.join(input_folder, "*.nc")))
print(f"🔎 Found {len(nc_files)} ERA5 files to process.\n")

for file_path in nc_files:
    file_name = os.path.basename(file_path)
    print(f"{'='*70}")
    print(f"📄 CHECKING ERA5 FILE: {file_name}")
    print(f"{'='*70}")

    try:
        # Open dataset without loading data array values into memory
        ds = xr.open_dataset(file_path)

        # --- Metadata check ---
        print(f"✅ NetCDF Version/Conventions: {ds.attrs.get('Conventions', 'N/A')}")

        # --- VARIABLES & UNITS ---
        print("\n📊 VARIABLES FOUND:")
        for var in ds.data_vars:
            unit = ds[var].attrs.get('units', 'N/A')
            long_name = ds[var].attrs.get('long_name', 'No description')
            print(f"  -> {var:12} | Units: {unit:10} | Shape: {ds[var].shape} | Size: {ds[var].size:,}")
            print(f"     Description: {long_name}")

        print("-" * 50)

        # --- DYNAMIC TIME INFO CHECK ---
        # Detects whether Copernicus named the time variable 'time' or 'valid_time'
        time_name = 'time' if 'time' in ds.coords else ('valid_time' if 'valid_time' in ds.coords else None)

        if time_name:
            start_t = ds[time_name].values[0]
            end_t = ds[time_name].values[-1]
            print(f"📅 TIME STEPS:  {len(ds[time_name])}")
            print(f"📅 DATE RANGE:  {start_t} to {end_t}")
        else:
            print("⚠️ TIME COORDINATE NOT FOUND! Check dataset structure.")

        # --- DYNAMIC GEO INFO CHECK ---
        lat_name = 'latitude' if 'latitude' in ds.coords else ('lat' if 'lat' in ds.coords else None)
        lon_name = 'longitude' if 'longitude' in ds.coords else ('lon' if 'lon' in ds.coords else None)

        if lat_name and lon_name:
            print(f"📍 LAT RANGE:   {ds[lat_name].values.min():.2f}° to {ds[lat_name].values.max():.2f}°")
            print(f"📍 LON RANGE:   {ds[lon_name].values.min():.2f}° to {ds[lon_name].values.max():.2f}°")

            # Check grid resolution spacing
            if len(ds[lat_name].values) > 1:
                res_lat = abs(ds[lat_name].values[1] - ds[lat_name].values[0])
                print(f"📏 GRID RES:    {res_lat:.2f}° (Expected ~0.25° for ERA5 reanalysis)")
        else:
            print("⚠️ GEOGRAPHICAL COORDINATES NOT FOUND!")

        ds.close()

    except Exception as e:
        print(f"❌ ERROR reading {file_name}: {e}")

    print("\n")

print("--- FINISHED SCANNING ---")

🔎 Found 48 ERA5 files to process.

📄 CHECKING ERA5 FILE: era5_2dew_2010.nc
✅ NetCDF Version/Conventions: CF-1.7

📊 VARIABLES FOUND:
  -> d2m          | Units: K          | Shape: (2920, 52, 52) | Size: 7,895,680
     Description: 2 metre dewpoint temperature
--------------------------------------------------
📅 TIME STEPS:  2920
📅 DATE RANGE:  2010-01-01T00:00:00.000000000 to 2010-12-31T21:00:00.000000000
📍 LAT RANGE:   33.00° to 45.75°
📍 LON RANGE:   17.00° to 29.75°
📏 GRID RES:    0.25° (Expected ~0.25° for ERA5 reanalysis)


📄 CHECKING ERA5 FILE: era5_2dew_2011.nc
✅ NetCDF Version/Conventions: CF-1.7

📊 VARIABLES FOUND:
  -> d2m          | Units: K          | Shape: (2920, 52, 52) | Size: 7,895,680
     Description: 2 metre dewpoint temperature
--------------------------------------------------
📅 TIME STEPS:  2920
📅 DATE RANGE:  2011-01-01T00:00:00.000000000 to 2011-12-31T21:00:00.000000000
📍 LAT RANGE:   33.00° to 45.75°
📍 LON RANGE:   17.00° to 29.75°
📏 GRID RES:    0.25° (Expected

### **Preprocessing ERA5**

In [ ]:
# ==============================================================================
# ERA5 STANDARDIZATION PIPELINE WITH METADATA & 4D SQUEEZE (DEEP LEARNING READY)
# ==============================================================================
import xarray as xr
import pandas as pd
import os
import shutil

# --- PATH SETTINGS ---
input_dir = "/content/drive/MyDrive/ERA5_data"
output_dir = "/content/drive/MyDrive/ERA5_data/final_standardized"
temp_dir = "/content/temp_processing"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(temp_dir, exist_ok=True)

years = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]

# Exact encoding match with CERRA + Added Dewpoint & T850 channels
my_encoding = {
    't2m':  {'zlib': True, 'complevel': 5, 'chunksizes': (1, 52, 52)},
    'u10':  {'zlib': True, 'complevel': 5, 'chunksizes': (1, 52, 52)},
    'v10':  {'zlib': True, 'complevel': 5, 'chunksizes': (1, 52, 52)},
    'd2m':  {'zlib': True, 'complevel': 5, 'chunksizes': (1, 52, 52)},
    't850': {'zlib': True, 'complevel': 5, 'chunksizes': (1, 52, 52)},
    'time': {
        'units': 'hours since 1900-01-01 00:00:00',
        'calendar': 'proleptic_gregorian'
    }
}

for year in years:
    # Target the combined files you created to get all 5 channels
    file_name = f"era5_combined_raw_{year}.nc"
    drive_path = os.path.join(input_dir, file_name)
    local_path = os.path.join(temp_dir, file_name)

    if not os.path.exists(drive_path):
        print(f"⚠️ Warning: {file_name} not found on Drive. Skipping...")
        continue

    print(f"Processing ERA5 {year}...")

    # 1. Copy to local disk (Critical for speed in Colab)
    shutil.copy(drive_path, local_path)

    # 2. Open Dataset
    ds = xr.open_dataset(local_path)

    # 3. FIX 4D TO 3D: Squeeze out the singleton level dimension from 't'
    ds = ds.squeeze(drop=True)

    # 4. Rename variables to clean standard names
    # Renaming 't' to 't850' to distinguish from t2m, and safety check for valid_time
    rename_vars = {'valid_time': 'time', 't': 't850'}
    ds = ds.rename({k: v for k, v in rename_vars.items() if k in ds.variables or k in ds.coords})

    # Rename lat/lon dimensions to match CERRA conventions
    rename_coords = {'lat': 'latitude', 'lon': 'longitude'}
    ds = ds.rename({k: v for k, v in rename_coords.items() if k in ds.variables or k in ds.coords})

    # 5. Sample check (Leap year check)
    is_leap = (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)
    samples = 2928 if is_leap else 2920

    if len(ds.time) != samples:
        raise ValueError(f"🚨 Error: Year {year} has {len(ds.time)} steps instead of {samples}!")

    # 6. Overwrite time axis for perfect sync (00:00, 03:00, etc.)
    ds['time'] = pd.date_range(start=f"{year}-01-01 00:00:00", periods=samples, freq='3H')

    # 7. Inject CF-1.8 Compliance Metadata for New Variables
    ds.attrs['Conventions'] = 'CF-1.8'
    ds.attrs['title'] = f'Standardized ERA5 Predictors for Greece ({year})'

    if 'd2m' in ds.data_vars:
        ds['d2m'].attrs = {
            'standard_name': 'dew_point_temperature',
            'long_name': '2 metre dewpoint temperature',
            'units': 'K'
        }
    if 't850' in ds.data_vars:
        ds['t850'].attrs = {
            'standard_name': 'air_temperature',
            'long_name': 'Temperature at 850 hPa pressure level',
            'units': 'K'
        }

    # 8. Save back to Drive in standard format
    save_path = os.path.join(output_dir, f"ERA5_Greece_{year}_52x52.nc")
    ds.to_netcdf(save_path, encoding=my_encoding)

    # Cleanup local files
    ds.close()
    os.remove(local_path)

    print(f"✅ Finished {year}!")

print("\n🚀 All ERA5 files are standardized to CF-1.8 (Squeezed T850, Dewpoint added, 52x52).")

In [ ]:
# ==============================================================================
# ERA5 STANDARDIZATION PIPELINE WITH METADATA & 4D SQUEEZE (DEEP LEARNING READY)
# ==============================================================================
import xarray as xr
import pandas as pd
import os
import shutil

# --- PATH SETTINGS ---
input_dir = "/content/drive/MyDrive/ERA5_data"
output_dir = "/content/drive/MyDrive/ERA5_data/final_standardized"
temp_dir = "/content/temp_processing"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(temp_dir, exist_ok=True)

years = [2010]

# Exact encoding match with CERRA + Added Dewpoint & T850 channels
my_encoding = {
    't2m':  {'zlib': True, 'complevel': 5, 'chunksizes': (1, 52, 52)},
    'u10':  {'zlib': True, 'complevel': 5, 'chunksizes': (1, 52, 52)},
    'v10':  {'zlib': True, 'complevel': 5, 'chunksizes': (1, 52, 52)},
    'd2m':  {'zlib': True, 'complevel': 5, 'chunksizes': (1, 52, 52)},
    't850': {'zlib': True, 'complevel': 5, 'chunksizes': (1, 52, 52)},
    'time': {
        'units': 'hours since 1900-01-01 00:00:00',
        'calendar': 'proleptic_gregorian'
    }
}

for year in years:
    # Target the combined files you created to get all 5 channels
    file_name = f"era5_combined_raw_{year}.nc"
    drive_path = os.path.join(input_dir, file_name)
    local_path = os.path.join(temp_dir, file_name)

    if not os.path.exists(drive_path):
        print(f"⚠️ Warning: {file_name} not found on Drive. Skipping...")
        continue

    print(f"Processing ERA5 {year}...")

    # 1. Copy to local disk (Critical for speed in Colab)
    shutil.copy(drive_path, local_path)

    # 2. Open Dataset
    ds = xr.open_dataset(local_path)

    # 3. FIX 4D TO 3D: Squeeze out the singleton level dimension from 't'
    ds = ds.squeeze(drop=True)

    # 4. Rename variables to clean standard names
    # Renaming 't' to 't850' to distinguish from t2m, and safety check for valid_time
    rename_vars = {'valid_time': 'time', 't': 't850'}
    ds = ds.rename({k: v for k, v in rename_vars.items() if k in ds.variables or k in ds.coords})

    # Rename lat/lon dimensions to match CERRA conventions
    rename_coords = {'lat': 'latitude', 'lon': 'longitude'}
    ds = ds.rename({k: v for k, v in rename_coords.items() if k in ds.variables or k in ds.coords})

    # 5. Sample check (Leap year check)
    is_leap = (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)
    samples = 2928 if is_leap else 2920

    if len(ds.time) != samples:
        raise ValueError(f"🚨 Error: Year {year} has {len(ds.time)} steps instead of {samples}!")

    # 6. Overwrite time axis for perfect sync (00:00, 03:00, etc.)
    ds['time'] = pd.date_range(start=f"{year}-01-01 00:00:00", periods=samples, freq='3H')

    # 7. Inject CF-1.8 Compliance Metadata for New Variables
    ds.attrs['Conventions'] = 'CF-1.8'
    ds.attrs['title'] = f'Standardized ERA5 Predictors for Greece ({year})'

    if 'd2m' in ds.data_vars:
        ds['d2m'].attrs = {
            'standard_name': 'dew_point_temperature',
            'long_name': '2 metre dewpoint temperature',
            'units': 'K'
        }
    if 't850' in ds.data_vars:
        ds['t850'].attrs = {
            'standard_name': 'air_temperature',
            'long_name': 'Temperature at 850 hPa pressure level',
            'units': 'K'
        }

    # 8. Save back to Drive in standard format
    save_path = os.path.join(output_dir, f"ERA5_Greece_{year}_52x52.nc")
    ds.to_netcdf(save_path, encoding=my_encoding)

    # Cleanup local files
    ds.close()
    os.remove(local_path)

    print(f"✅ Finished {year}!")

print("\n🚀 All ERA5 files are standardized to CF-1.8 (Squeezed T850, Dewpoint added, 52x52).")

Processing ERA5 2010...


/tmp/ipykernel_6248/4156777521.py:69: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  ds['time'] = pd.date_range(start=f"{year}-01-01 00:00:00", periods=samples, freq='3H')


✅ Finished 2010!

🚀 All ERA5 files are standardized to CF-1.8 (Squeezed T850, Dewpoint added, 52x52).


# **PREPARING CERRA & ERA5 TENSORS FOR DEEP LEARNING (B, C, H, W) FORMAT**

### **ERA5 Concatenation**

In [ ]:
# ==============================================================================
# ERA5 FULL TIME-SERIES CONCATENATION (2010-2021) - FIXED GEOGRAPHY
# ==============================================================================
import os
import glob
import xarray as xr
import shutil

# Paths
era5_input = "/content/drive/MyDrive/ERA5_data/final_standardized"
era5_final_dir = "/content/drive/MyDrive/ERA5_data/combined"
os.makedirs(era5_final_dir, exist_ok=True)

print("🔗 Gathering and sorting all ERA5 yearly files...")

# 1. Get all file paths and sort them strictly by name/year to ensure chronological order
file_list = sorted(glob.glob(os.path.join(era5_input, "ERA5_Greece_*_52x52.nc")))

if len(file_list) == 0:
    raise FileNotFoundError(f"🚨 No standardized files found in {era5_input}!")

print(f"📦 Found {len(file_list)} files. Combining now...")

# ==============================================================================
# 2. FIX: Open multiple files by strictly concatenating ONLY along the time axis
# This stops xarray from automatically reversing your latitude grid!
# ==============================================================================
ds_era5 = xr.open_mfdataset(
    file_list,
    combine='nested',
    concat_dim='time',
    chunks={'time': 500}
)

# 3. Verification of total samples
total_samples = len(ds_era5.time)
print(f"📊 Total ERA5 Concat Samples: {total_samples}")

# 4. Define Compression Encoding for the massive combined file
save_encoding = {}
for var in ds_era5.data_vars:
    save_encoding[var] = {
        'zlib': True,
        'complevel': 5,
        'chunksizes': (1, 52, 52)
    }
save_encoding['time'] = {
    'units': 'hours since 1900-01-01 00:00:00',
    'calendar': 'proleptic_gregorian'
}

# 5. Save to Drive via local temp for maximum I/O speed and stability
temp_path = "/content/ERA5_2010_2021_combined.nc"
final_drive_path = os.path.join(era5_final_dir, "ERA5_2010_2021_combined.nc")

print("💾 Writing combined NetCDF to local disk (this might take a minute)...")
ds_era5.to_netcdf(temp_path, encoding=save_encoding)

print("🚚 Moving final combined file to Google Drive...")
shutil.move(temp_path, final_drive_path)

# Close dataset to free up RAM
ds_era5.close()

print(f"✅ Done! Full dataset is safe and geolocated correctly at: {final_drive_path}")

🔗 Gathering and sorting all ERA5 yearly files...
📦 Found 12 files. Combining now...
📊 Total ERA5 Concat Samples: 35064
💾 Writing combined NetCDF to local disk (this might take a minute)...
🚚 Moving final combined file to Google Drive...
✅ Done! Full dataset is safe and geolocated correctly at: /content/drive/MyDrive/ERA5_data/combined/ERA5_2010_2021_combined.nc


### **Cerra Concatenation**

In [ ]:
# ==============================================================================
# CERRA TIME-SERIES CONCATENATION (STRICT YEARLY FILTERING)
# ==============================================================================
import os
import glob
import xarray as xr
import shutil

# Paths
cerra_input = "/content/drive/MyDrive/CERRA_processing/final_standardized"
cerra_final_dir = "/content/drive/MyDrive/CERRA_processing"
os.makedirs(cerra_final_dir, exist_ok=True)

print("🔗 Gathering and sorting CERRA yearly files...")


file_list = sorted(glob.glob(os.path.join(cerra_input, "CERRA_Greece_2*.nc")))

if len(file_list) == 0:
    raise FileNotFoundError(f"🚨 No yearly CERRA files found matching the pattern in {cerra_input}!")

print(f"📦 Found {len(file_list)} yearly files to combine. (Static files ignored successfully).")
print(f"📋 First file: {os.path.basename(file_list[0])}")
print(f"📋 Last file:  {os.path.basename(file_list[-1])}")

# Open multiple files as one dataset
ds_cerra = xr.open_mfdataset(
    file_list,
    combine='nested',
    concat_dim='time',
    chunks={'time': 500}
)

# Verification of samples
total_samples = len(ds_cerra.time)
print(f"📊 Total Cerra Samples: {total_samples}")

# Save to Drive (via local temp for stability)
temp_path = "/content/Cerra_2010_2021_combined.nc"

save_encoding = {
    't2m': {
        'zlib': True,
        'complevel': 5,
        'chunksizes': (1, len(ds_cerra.latitude), len(ds_cerra.longitude))
    }
}
if 'time' in ds_cerra.coords:
    save_encoding['time'] = {
        'units': 'hours since 1900-01-01 00:00:00',
        'calendar': 'proleptic_gregorian'
    }

ds_cerra.to_netcdf(temp_path, encoding=save_encoding)
shutil.move(temp_path, os.path.join(cerra_final_dir, "Cerra_2010_2021_combined.nc"))

ds_cerra.close()
print("✅ Cerra combined file is now in your Drive, clean from static fields!")

🔗 Gathering and sorting CERRA yearly files...
📦 Found 12 yearly files to combine. (Static files ignored successfully).
📋 First file: CERRA_Greece_2010.nc
📋 Last file:  CERRA_Greece_2021.nc
📊 Total Cerra Samples: 35064
✅ Cerra combined file is now in your Drive, clean from static fields!


In [ ]:
# ==============================================================================
# AUTOMATIC CERRA STATIC FIELDS CONCATENATION
# ==============================================================================
import os
import glob
import xarray as xr
import shutil

# Φάκελος εισόδου και εξόδου
cerra_input_dir = "/content/drive/MyDrive/CERRA_processing/final_standardized"
cerra_main_dir = "/content/drive/MyDrive/CERRA_processing"
final_static_path = os.path.join(cerra_main_dir, "CERRA_Greece_Static_CHW.nc")

print("🔎 Searching automatically for static files...")

# Ψάχνουμε αυτόματα με patterns για να μην κολλάμε στα ακριβή ονόματα
orog_files = glob.glob(os.path.join(cerra_input_dir, "*orog*"))
lsm_files = glob.glob(os.path.join(cerra_input_dir, "*land*")) + glob.glob(os.path.join(cerra_input_dir, "*lsm*"))

if not orog_files or not lsm_files:
    # Αν όντως λείπει κάτι, τυπώνουμε τι υπάρχει μέσα για να το δεις
    all_files = os.listdir(cerra_input_dir)
    raise FileNotFoundError(f"🚨 Κάτι λείπει! Μέσα στον φάκελο υπάρχουν μόνο αυτά: {all_files}")

# Παίρνουμε το πρώτο αρχείο που ταίριαξε για κάθε κατηγορία
found_orog_path = orog_files[0]
found_lsm_path = lsm_files[0]

print(f"📦 Found Orography: {os.path.basename(found_orog_path)}")
print(f"📦 Found Land-Sea Mask: {os.path.basename(found_lsm_path)}")
print("🔗 Joining static fields as they are...")

# 1. Open the automatically detected datasets
ds_orog = xr.open_dataset(found_orog_path)
ds_lsm = xr.open_dataset(found_lsm_path)

# 2. Merge them into one dataset
ds_combined = xr.merge([ds_orog, ds_lsm])

# 3. Get the exact variable names as they are inside the files
v_orog = [v for v in ds_combined.data_vars if 'orog' in v][0]
v_lsm = [v for v in ds_combined.data_vars if 'lsm' in v or 'land' in v][0]

da_stacked = ds_combined[[v_orog, v_lsm]].to_array(dim='channel')

ds_static_final = da_stacked.transpose('channel', 'latitude', 'longitude').to_dataset(name='static_data')

print(f"📐 Shape: {ds_static_final.static_data.shape}")

temp_static = "/content/temp_static.nc"
ds_static_final.to_netcdf(temp_static)
shutil.move(temp_static, final_static_path)

ds_orog.close()
ds_lsm.close()
print(f"✅ Done! Combined static file saved at: {final_static_path}")

### Data Reshaping to BCHW format

In [ ]:
# ==============================================================================
# ERA5 TENSOR TRANSFORMATION TO BCHW FORMAT (ALL 5 CHANNELS)
# ==============================================================================
import xarray as xr
import shutil
import os

combined_file = "/content/drive/MyDrive/ERA5_data/combined/ERA5_2010_2021_combined.nc"
final_bchw = "/content/drive/MyDrive/ERA5_data/combined/ERA5_2010_2021_BCHW.nc"

print("🎯 Transforming ERA5 into BCHW with 5 Deep Learning Channels...")

# Open the full dataset
ds = xr.open_dataset(combined_file)

# Προσθέτουμε ΚΑΙ το d2m ΚΑΙ το t850 που φτιάξαμε στο standardization
variables = ['t2m', 'u10', 'v10', 'd2m', 't850']
da_stacked = ds[variables].to_array(dim='channel')

# Transpose strictly to (Batch/Time, Channel, Height, Width)
ds_final = da_stacked.transpose('time', 'channel', 'latitude', 'longitude').to_dataset(name='era5_data')

# Θα σου τυπώσει (35064, 5, 52, 52) αν έχεις όλη την 12ετία
print(f"📏 Final Tensor Shape: {ds_final.era5_data.shape}")

# Optimized encoding to keep the file compressed and fast to load in PyTorch/Keras
my_encoding = {
    'era5_data': {
        'zlib': True,
        'complevel': 5,
        'chunksizes': (1, 5, 52, 52) # Optimized for single batch step extraction
    }
}

# Save via local temp for maximum I/O stability
temp_path = "/content/temp_era5_bchw.nc"
ds_final.to_netcdf(temp_path, encoding=my_encoding)
shutil.move(temp_path, final_bchw)

ds.close()
print("✅ ERA5 BCHW dataset is officially ready for the UNet!")

🎯 Transforming ERA5 into BCHW with 5 Deep Learning Channels...
📏 Final Tensor Shape: (35064, 5, 52, 52)
✅ ERA5 BCHW dataset is officially ready for the UNet!


In [ ]:
import xarray as xr
import shutil
import os

combined_file = "/content/drive/MyDrive/CERRA_processing/Cerra_2010_2021_combined.nc"
final_bchw = "/content/drive/MyDrive/CERRA_processing/Cerra_2010_2021_BCHW.nc"

print("🎯 Transforming CERRA into BCHW (RAM-optimized)...")

ds = xr.open_dataset(combined_file, chunks={'time': 100})

da_bchw = ds['t2m'].expand_dims(dim='channel', axis=1)

ds_final = da_bchw.to_dataset(name='cerra_data')

print(f"📏 Target Shape (B,C,H,W): {ds_final.cerra_data.shape}")

print("💾 Writing to local disk (processing chunks)...")
ds_final.to_netcdf("/content/temp_cerra.nc")

print("🚚 Moving to Drive...")
shutil.move("/content/temp_cerra.nc", final_bchw)

print("✅ CERRA BCHW ready!")

🎯 Transforming CERRA into BCHW (RAM-optimized)...


/tmp/ipykernel_3023/4031250769.py:10: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 100. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(combined_file, chunks={'time': 100})


📏 Target Shape (B,C,H,W): (35064, 1, 256, 256)
💾 Writing to local disk (processing chunks)...
🚚 Moving to Drive...
✅ CERRA BCHW ready!


In [ ]:
# ==============================================================================
# CERRA STATIC FIELDS - CONCATENATION WITH B=1 (BCHW READY)
# ==============================================================================
import os
import glob
import xarray as xr
import shutil

cerra_input_dir = "/content/drive/MyDrive/CERRA_processing/final_standardized"
cerra_main_dir = "/content/drive/MyDrive/CERRA_processing"
final_static_path = os.path.join(cerra_main_dir, "CERRA_Greece_Static_B1_CHW.nc")

print("🔎 Searching for static files...")
orog_files = glob.glob(os.path.join(cerra_input_dir, "*orog*"))
lsm_files = glob.glob(os.path.join(cerra_input_dir, "*land*")) + glob.glob(os.path.join(cerra_input_dir, "*lsm*"))

if not orog_files or not lsm_files:
    raise FileNotFoundError("🚨 Missing static files in the directory!")

ds_orog = xr.open_dataset(orog_files[0])
ds_lsm = xr.open_dataset(lsm_files[0])

ds_combined = xr.merge([ds_orog, ds_lsm])
v_orog = [v for v in ds_combined.data_vars if 'orog' in v][0]
v_lsm = [v for v in ds_combined.data_vars if 'lsm' in v or 'land' in v][0]
da_stacked = ds_combined[[v_orog, v_lsm]].to_array(dim='channel')

if 'time' in da_stacked.dims:
    da_stacked = da_stacked.squeeze(dim='time', drop=True)

da_padded = da_stacked.expand_dims(dim={'time': 1}, axis=0)

ds_static_final = da_padded.transpose('time', 'channel', 'latitude', 'longitude').to_dataset(name='static_data')

print(f"📐 Shape: {ds_static_final.static_data.shape}")

temp_static = "/content/temp_static_b1.nc"
ds_static_final.to_netcdf(temp_static)
shutil.move(temp_static, final_static_path)

ds_orog.close()
ds_lsm.close()
print(f"✅ Done! Combined static file with B=1 saved at: {final_static_path}")

🔎 Searching for static files...
📐 Shape: (1, 2, 256, 256)
✅ Done! Combined static file with B=1 saved at: /content/drive/MyDrive/CERRA_processing/CERRA_Greece_Static_B1_CHW.nc


# Distribution Analysis

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pandas as pd
from typing import Optional, Tuple, Dict, List


In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from typing import Dict, Tuple
import os

class Variable_Statistical_Analysis:
    def __init__(self, input_path: Path, output_path: Path, variable: str="t2m"):
        self.input_path = Path(input_path)
        self.output_path = Path(output_path)
        self.variable = variable
        self.regions = ["Full_Domain"]

        os.makedirs(self.output_path, exist_ok=True)

    def variable_values(self, data: xr.Dataset) -> np.ndarray:
        chunk = 50 # Μεγαλύτερο chunk για ταχύτητα
        time_chunks = np.arange(0, len(data.time), chunk)
        print(f" Processing {len(time_chunks)} chunks for {self.variable} (Full Dataset Mode)")

        # Πρώτο πέρασμα μόνο για μέτρηση μεγέθους (για να δεσμεύσουμε μνήμη μία φορά)
        num_values = 0
        for start in time_chunks:
            end = min(start + chunk, len(data.time))
            # count non-nan values
            num_values += int(data[self.variable].isel(time=slice(start, end)).count())

        # Δέσμευση πίνακα για ΟΛΑ τα δεδομένα
        total_values = np.empty(num_values, dtype=np.float32)
        idx = 0

        # Δεύτερο πέρασμα: Συλλογή όλων των τιμών (Χωρίς Clipping)
        for start in time_chunks:
            end = min(start + chunk, len(data.time))
            samples_chunk = data[self.variable].isel(time=slice(start, end)).values.flatten()
            valid_chunk = samples_chunk[~np.isnan(samples_chunk)]

            if len(valid_chunk) > 0:
                end_idx = idx + len(valid_chunk)
                total_values[idx:end_idx] = valid_chunk
                idx = end_idx

        return total_values

    def statistics(self, values_full: np.ndarray, chunk_size: int=10000) -> Dict:
        print(f" Calculating statistics from {len(values_full)} points...")
        count = len(values_full)
        mean_val = np.mean(values_full)

        # Υπολογισμός των p01 και p999 από ΟΛΟ το dataset
        p01 = np.percentile(values_full, 0.1)
        p999 = np.percentile(values_full, 99.9)

        var_min, var_max = float(values_full.min()), float(values_full.max())

        # Υπολογισμός SE για STD
        se = 0.0
        for i in range(0, count, chunk_size):
            end = min(i + chunk_size, count)
            schunk = values_full[i:end]
            se += np.sum((schunk - mean_val)**2)

        std = np.sqrt(se / count) if count > 1 else 0.0

        # Δείγμα μόνο για skewness/kurtosis για να μην "σκάσει" η scipy
        sample_for_moments = np.random.choice(values_full, min(200000, count), replace=False)

        return {
            'count': count,
            'mean': mean_val,
            'std': std,
            'min': var_min,
            'max': var_max,
            'p01': p01,      # Το ζήτησες από όλο το dataset
            'p999': p999,    # Το ζήτησες από όλο το dataset
            'median': np.median(values_full), # Από όλο το dataset
            'q25': np.percentile(values_full, 25),
            'q75': np.percentile(values_full, 75),
            'skewness': stats.skew(sample_for_moments),
            'kurtosis': stats.kurtosis(sample_for_moments)
        }

    def statistics_table(self, stats_dict: Dict) -> None:
        df = pd.DataFrame([stats_dict], index=[self.variable])
        csv_path = self.output_path / f'{self.variable}_statistics.csv'
        df.round(4).to_csv(csv_path)
        print(f"✓ Statistics saved: {csv_path}")

    def create_simple_plot(self, data: np.ndarray) -> None:
        """Plot μόνο για το raw distribution αφού δεν θέλουμε κανονικοποίηση"""
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        fig.suptitle(f'{self.variable.upper()} Raw Distribution Analysis', fontsize=16)

        # Histogram
        axes[0].hist(data[::20], bins=100, color='skyblue', density=True)
        axes[0].set_title("Raw Distribution (Full Dataset)")

        # Boxplot
        axes[1].boxplot(data[::100], patch_artist=True)
        axes[1].set_title("Outliers (No Clipping)")

        plt.savefig(self.output_path / f'{self.variable}_analysis.png', dpi=200)
        plt.close()
        print(f"✓ Plot saved in {self.output_path}")

    # Η normalize_variable παραμένει αλλά δεν χρησιμοποιείται για να μην αλλάξει η δομή του κώδικα
    def normalize_variable(self, values_chunk: np.ndarray) -> Tuple[np.ndarray, float, float]:
        v_min, v_max = values_chunk.min(), values_chunk.max()
        normalized = (values_chunk - v_min) / (v_max - v_min) if v_max > v_min else values_chunk
        return normalized, v_min, v_max

In [ ]:
import xarray as xr
import numpy as np
import gc
from pathlib import Path

# --- CONFIGURATION ---
output_base = Path('/content/drive/MyDrive/Final_Datasets')
era5_path = Path('/content/drive/MyDrive/ERA5_data/combined/ERA5_2014_2021_BCHW.nc')
orog_path = Path('/content/drive/MyDrive/CERRA_processing/Cerra_Orography_BCHW.nc')

output_base.mkdir(parents=True, exist_ok=True)

# ERA5 Channels ONLY
era5_channels = {0: 't2m', 1: 'u10', 2: 'v10'}

def run_full_analysis():
    # --- 1. ERA5 STATISTICAL ANALYSIS ---
    print("\n>>> Starting ERA5 Full Statistical Analysis (No Clip, No Norm)...")
    ds_era = xr.open_dataset(era5_path)
    era_internal_name = list(ds_era.data_vars)[0]

    for ch_idx, var_name in era5_channels.items():
        print(f" Processing ERA5 Channel: {var_name}")
        out_path = output_base / 'ERA5' / var_name

        # Απομόνωση καναλιού
        ds_var = ds_era.isel(channel=ch_idx).rename({era_internal_name: var_name})

        # Διόρθωση: Περνάμε και το input_path και το output_path
        analyzer = Variable_Statistical_Analysis(
            input_path=era5_path,
            output_path=out_path,
            variable=var_name
        )

        # Παίρνουμε τις RAW τιμές (χωρίς clip)
        data_raw = analyzer.variable_values(ds_var)

        # Υπολογισμός στατιστικών (p01/p999 από όλο το data)
        stats = analyzer.statistics(data_raw)
        analyzer.statistics_table(stats)

        # Plot μόνο της RAW κατανομής
        analyzer.create_simple_plot(data_raw)

        del data_raw
        gc.collect()

    ds_era.close()

    # --- 2. OROGRAPHY STATISTICAL ANALYSIS ---
    print("\n>>> Starting Orography Statistical Analysis...")
    ds_orog = xr.open_dataset(orog_path)
    orog_internal_name = list(ds_orog.data_vars)[0]

    ds_var_orog = ds_orog.isel(channel=0).rename({orog_internal_name: 'orog'})

    # Διόρθωση και εδώ στα ορίσματα
    analyzer_orog = Variable_Statistical_Analysis(
        input_path=orog_path,
        output_path=output_base / 'Orography',
        variable='orog'
    )

    data_raw_orog = analyzer_orog.variable_values(ds_var_orog)
    stats = analyzer_orog.statistics(data_raw_orog)
    analyzer_orog.statistics_table(stats)
    analyzer_orog.create_simple_plot(data_raw_orog)

    ds_orog.close()

    print("\n🚀 ANALYSIS FINISHED! CERRA was skipped.")

# Run
if __name__ == "__main__":
    run_full_analysis()


>>> Starting ERA5 Full Statistical Analysis (No Clip, No Norm)...
 Processing ERA5 Channel: t2m
 Processing 468 chunks for t2m (Full Dataset Mode)
 Calculating statistics from 63208704 points...
✓ Statistics saved: /content/drive/MyDrive/Final_Datasets/ERA5/t2m/t2m_statistics.csv
✓ Plot saved in /content/drive/MyDrive/Final_Datasets/ERA5/t2m
 Processing ERA5 Channel: u10
 Processing 468 chunks for u10 (Full Dataset Mode)
 Calculating statistics from 63208704 points...
✓ Statistics saved: /content/drive/MyDrive/Final_Datasets/ERA5/u10/u10_statistics.csv
✓ Plot saved in /content/drive/MyDrive/Final_Datasets/ERA5/u10
 Processing ERA5 Channel: v10
 Processing 468 chunks for v10 (Full Dataset Mode)
 Calculating statistics from 63208704 points...
✓ Statistics saved: /content/drive/MyDrive/Final_Datasets/ERA5/v10/v10_statistics.csv
✓ Plot saved in /content/drive/MyDrive/Final_Datasets/ERA5/v10

>>> Starting Orography Statistical Analysis...
 Processing 1 chunks for orog (Full Dataset Mode)
 

#Split data

In [ ]:
import xarray as xr
from pathlib import Path

era5_ds = xr.open_dataset('ERA5_2014_2021_Clipped.nc')
cerra_ds = xr.open_dataset('Cerra_2014_2021_Clipped.nc')
orog_ds = xr.open_dataset('Cerra_Orography_BCHW.nc')

def split_and_save(ds, name):
    print(f"Splitting {name}...")

    # Train: 2014 - 2020
    train = ds.sel(time=slice('2014-01-01', '2020-12-31'))

    # Validation: Πρώτο μισό του 2021
    val = ds.sel(time=slice('2021-01-01', '2021-06-30'))

    # Test: Δεύτερο μισό του 2021
    test = ds.sel(time=slice('2021-07-01', '2021-12-31'))

    # Αποθήκευση
    train.to_netcdf(f'{name}_train.nc')
    val.to_netcdf(f'{name}_val.nc')
    test.to_netcdf(f'{name}_test.nc')

    print(f"✓ {name} split complete!")
    return train, val, test

# Εκτέλεση του split
train_era, val_era, test_era = split_and_save(era5_ds, 'ERA5')
train_cer, val_cer, test_cer = split_and_save(cerra_ds, 'CERRA')

# Έλεγχος μεγεθών
print(f"\nTrain steps: {len(train_era.time)}")
print(f"Val steps: {len(val_era.time)}")
print(f"Test steps: {len(test_era.time)}")

Min Temp: 243.62 K
Max Temp: 317.69 K


In [ ]:
!ls -R /content/drive/MyDrive/CERRA_dataset_final/

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import h5py
import xarray as xr
import json
import numpy as np
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import h5py
import xarray as xr
import json
import numpy as np
from pathlib import Path

class GreeceDownscalingDataset(Dataset):
    def __init__(self, era5_path, cerra_path, static_path, stats_path):
        # 1. Φόρτωση στατιστικών
        with open(stats_path, 'r') as f:
            self.stats = json.load(f)

        # 2. ERA5 & CERRA Files
        self.era5_file = h5py.File(era5_path, 'r')
        self.era5_data = self.era5_file['data']

        self.cerra_ds = xr.open_dataset(cerra_path)
        self.cerra_t2m = self.cerra_ds.t2m.squeeze('region')

        # 3. STATIC (Orography)
        ds_static = xr.open_dataset(static_path)
        orog_values = ds_static.orography.values

        # Μετατροπή σε tensor
        orog_tensor = torch.from_numpy(orog_values).float()

        # --- ΕΔΩ ΤΟ ΦΤΙΑΧΝΟΥΜΕ ---
        #orog_tensor = torch.flip(orog_tensor, dims=[0])

        # Κανονικοποίηση Static (Πλέον χρησιμοποιούμε τη σωστή μεταβλητή)
        m_s = self.stats['static']['mean']
        s_s = self.stats['static']['std']
        self.static_norm = (orog_tensor - m_s) / s_s

        self.n_samples = self.era5_data.shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        # --- 1. ERA5 (Input) ---
        x_era5 = torch.from_numpy(self.era5_data[idx]).float().permute(2, 0, 1)

        # Κανονικοποίηση ERA5
        for c in range(3):
            m = self.stats['era5']['mean'][c]
            s = self.stats['era5']['std'][c]
            x_era5[c] = (x_era5[c] - m) / s

        # Upsampling
        x_upsampled = F.interpolate(x_era5.unsqueeze(0), size=(256, 256),
                                    mode='bilinear', align_corners=False).view(3, 256, 256)

        # --- 2. STATIC (Orography) ---
        s_norm = self.static_norm.view(1, 256, 256)

        # Συνένωση (4 κανάλια)
        x_final = torch.cat([x_upsampled, s_norm], dim=0)

        # --- 3. CERRA (Target) ---
        # Παίρνουμε το δείγμα
        y_raw = torch.from_numpy(self.cerra_t2m.isel(time=idx).values).float()

        # Flip Target αν είναι ανάποδα στο plot
        y_raw = torch.flip(y_raw, dims=[0])

        y = y_raw.view(1, 256, 256)

        # Κανονικοποίηση Target
        m_y = self.stats['cerra']['mean']
        s_y = self.stats['cerra']['std']
        y = (y - m_y) / s_y

        return x_final, y

IndentationError: unexpected indent (1288128542.py, line 99)

In [ ]:
import xarray as xr

path = "/content/drive/MyDrive/ERA5_data/combined/ERA5_2010_2021_combined.nc"
ds = xr.open_dataset(path)

print("\n===== DATASET =====")
print(ds)

print("\n===== VARIABLES =====")
for v in ds.data_vars:
    print(v, ds[v].dims, ds[v].shape)

print("\n===== ERA5 DATA DETAILS =====")
print("dims:", ds.dims)
print("coords:", list(ds.coords))


===== DATASET =====
<xarray.Dataset> Size: 2GB
Dimensions:    (time: 35064, latitude: 52, longitude: 52)
Coordinates:
  * time       (time) datetime64[ns] 281kB 2010-01-01 ... 2021-12-31T21:00:00
  * latitude   (latitude) float64 416B 45.75 45.5 45.25 45.0 ... 33.5 33.25 33.0
  * longitude  (longitude) float64 416B 17.0 17.25 17.5 ... 29.25 29.5 29.75
    number     int64 8B ...
    expver     (time) <U4 561kB ...
Data variables:
    u10        (time, latitude, longitude) float32 379MB ...
    v10        (time, latitude, longitude) float32 379MB ...
    t2m        (time, latitude, longitude) float32 379MB ...
    d2m        (time, latitude, longitude) float32 379MB ...
    t850       (time, latitude, longitude) float32 379MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.8
    institution:             European Centre for Medium-Range Weath

In [ ]:
import xarray as xr

path = "/content/drive/MyDrive/ERA5_data/combined/ERA5_2010_2021_combined.nc"
ds = xr.open_dataset(path)

print(ds)
lat = ds["latitude"].values
lon = ds["longitude"].values

print("Latitude start:", lat[0])
print("Latitude end:", lat[-1])
print("Longitude start:", lon[0])
print("Longitude end:", lon[-1])

KeyboardInterrupt: 

In [ ]:
import xarray as xr

# 1. Ορισμός των μονοπατιών για το αρχικό και το νέο αρχείο
input_file = "/content/drive/MyDrive/ERA5_data/combined/ERA5_2010_2021_combined.nc"
output_file = "/content/drive/MyDrive/ERA5_data/combined/ERA5_2010_2021_fixed.nc"

print("Φόρτωση του dataset...")
# Χρησιμοποιούμε chunks αν η μνήμη RAM του Colab ζορίζεται,
# αλλιώς ένα απλό xr.open_dataset(input_file) αρκεί.
ds = xr.open_dataset(input_file)

print("Αλλαγή της φοράς του latitude...")
# Τρόπος 1: Ταξινόμηση σε αύξουσα σειρά (από 33.0 έως 45.75)
ds_fixed = ds.sortby("latitude", ascending=True)

# Ή Τρόπος 2: Απλό reverse στην υπάρχουσα διάσταση
# ds_fixed = ds.isel(latitude=slice(None, None, -1))

print("Αποθήκευση στο νέο αρχείο NetCDF...")
# Αποθήκευση του διορθωμένου αρχείου
ds_fixed.to_netcdf(output_file)

print(f"Έτοιμο! Το διορθωμένο αρχείο αποθηκεύτηκε ως: {output_file}")

# Επιβεβαίωση της αλλαγής
print("\nΝέο Latitude αρχή και τέλος:")
print(ds_fixed.latitude.values[0], "έως", ds_fixed.latitude.values[-1])

Φόρτωση του dataset...


KeyboardInterrupt: 